# Introduction to h3xplorer

In [ ]:
from pathlib import Path

import polars as pl
from lonboard import Map, basemap

from h3xplorer import core, data, mapping
from h3xplorer.inputs.points import xy_data_to_wgs84
from h3xplorer.inputs.table import read_dataset

In [ ]:
data_dir = Path.cwd().parent / "tests" / "fixture_data"
xys = read_dataset(data_dir / "xy.parquet")
xys = xys.with_columns(pl.Series("population", [300, 200, 1000, 100, 600]))
latlons = xy_data_to_wgs84(xys, "x", "y", 27700)
df_hex_refs, hex_refs = data.get_hexagon_refs_for_points(latlons, 2)
display(df_hex_refs)
display(hex_refs)

In [ ]:
hexes = data.get_hexagon_polygons(hex_refs)
df_agg = data.groupby_ref_col(df_hex_refs, pop_sum={"column": "population", "agg": "sum"})
gdf = data.join_pldf_to_gdf(df_agg, hexes)
gdf

In [ ]:
layer = mapping.create_polygon_layer(gdf, "pop_sum")
Map([layer], basemap_style=basemap.CartoBasemap.DarkMatter)

In [ ]:
maps = []
for idx, hex_size in enumerate(range(5)):
    maps.append(core.xy_plot(data_dir / "xy.parquet", "x", "y", 27700, hex_size, "id", "mean"))
    display(maps[idx])